In [1]:
import pandas as pd



In [2]:
df=pd.read_excel(r"C:\Users\Acer\Downloads\translated_hindi_comments.xlsx")

In [3]:
df.head(3)

,Post,Labels Set
0,"""dr. bharat agravaal kaa kaalam: cheeni havaal...",normal
1,bharat kaa pantapradhaan pad kaa janmajaat umi...,fake
2,amariikii opan tennis men bharat ke rohan bopa...,normal


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9483 entries, 0 to 9482
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Post        9474 non-null   object
 1   Labels Set  9483 non-null   object
dtypes: object(2)
memory usage: 148.3+ KB


In [5]:
df=df.dropna()

In [6]:

print(df['Post'].isnull().sum()) 
print(df['Post'].apply(type).value_counts())  


0
Post
<class 'str'>    9474
Name: count, dtype: int64


In [7]:
df=df.dropna()

In [8]:
df['Labels Set'] = df['Labels Set'].replace({0: 'normal', 1: "hate"})

In [9]:
df = df[df['Labels Set'] != 'd']
df = df[df['Labels Set'] != 'fake']

In [10]:
df = df.drop_duplicates()

# svm-rbf

# svm-rbf-fasttext

In [11]:
import pandas as pd
import fasttext
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import numpy as np
import re

import re
def data_processing(Text):
    Text = re.sub(r'[^a-zA-Z]', ' ', Text).lower()
    words = Text.split()
    words = [word for word in words if len(word) >=3 ]
    return ' '.join(words)

# Clean the text in the dataset
df.Post = df['Post'].apply(data_processing)

from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
import re
def data_processing(Text):
    Text = re.sub(r'[^a-zA-Z]', ' ', Text).lower()
    words = Text.split()
    words = [lemmatizer.lemmatize(word) for word in words if len(word) >=3 ]
    return ' '.join(words)
df.Post = df['Post'].apply(data_processing)
import spacy
nlp = spacy.load('en_core_web_sm')
def remove_stopwords(text):
  doc = nlp(text)
  no_stopwords_list = [word.text for word in doc if not word.is_stop]
  return ' '.join(no_stopwords_list)

df['Post'] = df['Post'].apply(remove_stopwords)

# Split data into features and labels
X = df['Post']
y = df['Labels Set']

# Split dataset into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Save the training data to a temporary file for FastText
train_file = 'train.txt'
val_file = 'val.txt'

# Save training data for FastText
with open(train_file, 'w', encoding="utf-8") as f:
    for text, label in zip(X_train, y_train):
        f.write(f'__label__{label} {text}\n')

# Train FastText model
ft_model = fasttext.train_supervised(train_file)

# Get embeddings for the training and validation sets
def get_fasttext_embeddings(texts, model):
    embeddings = []
    for text in texts:
        # Get the vector representation for the entire text
        vec = model.get_sentence_vector(text)
        embeddings.append(vec)
    return np.array(embeddings)

X_train_embeds = get_fasttext_embeddings(X_train, ft_model)
X_val_embeds = get_fasttext_embeddings(X_val, ft_model)

# Train SVM with RBF kernel
svm_rbf = SVC(kernel='rbf', gamma='scale')  # Use 'scale' for gamma
svm_rbf.fit(X_train_embeds, y_train)

# Make predictions
y_pred = svm_rbf.predict(X_val_embeds)

# Evaluate the model
accuracy = accuracy_score(y_val, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print(classification_report(y_val, y_pred))


Accuracy: 0.88
              precision    recall  f1-score   support

        hate       0.85      0.83      0.84       617
      normal       0.90      0.91      0.91      1039

    accuracy                           0.88      1656
   macro avg       0.88      0.87      0.87      1656
weighted avg       0.88      0.88      0.88      1656



In [12]:
def predict_sentiment(text):
    # Get the embedding for the new text
    embedding = ft_model.get_sentence_vector(text)
    # Predict the label
    prediction = svm_rbf.predict([embedding])
    return prediction[0]

# Example input
input_text = "Sarkar banne ke bad Hindu hit me ek bhi faisla Jo bjp ke dwara liya gaya ho,bjp ko  gay,gobar,mandir,masjid aur nafrat faila kar vot chahiye"
predicted_label = predict_sentiment(input_text)

# Output the prediction
print(f"Predicted Label for the input text: {predicted_label}")

Predicted Label for the input text: normal


# fasttext_lstm

In [13]:
import pandas as pd
import fasttext
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import re



# Function to clean text

df['Labels Set'] = df['Labels Set'].replace({'hate': 1, 'normal': 0})

# Split data into features and labels
X = df['Post']
y = np.array(df['Labels Set'], dtype=np.float32)  # Ensure labels are in float32

# Split dataset into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Save the training data to a temporary file for FastText
train_file = 'train.txt'

with open(train_file, 'w', encoding="utf-8") as f:
    for text, label in zip(X_train, y_train):
        f.write(f'__label__{int(label)} {text}\n')  # Ensure label is an integer

# Train FastText model
ft_model = fasttext.train_supervised(train_file)

# Get embeddings for the training and validation sets
def get_fasttext_embeddings(texts, model):
    embeddings = []
    for text in texts:
        vec = model.get_sentence_vector(text)
        embeddings.append(vec)
    return np.array(embeddings)

# Generate embeddings
X_train_embeds = get_fasttext_embeddings(X_train, ft_model)
X_val_embeds = get_fasttext_embeddings(X_val, ft_model)

# Reshape embeddings for LSTM input
X_train_reshaped = X_train_embeds.reshape((X_train_embeds.shape[0], 1, X_train_embeds.shape[1]))
X_val_reshaped = X_val_embeds.reshape((X_val_embeds.shape[0], 1, X_val_embeds.shape[1]))

# Build LSTM model
model = Sequential()
model.add(LSTM(64, return_sequences=False, input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2])))  # Shape: (timesteps, features)
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train the model
model.fit(X_train_reshaped, y_train, epochs=10, validation_data=(X_val_reshaped, y_val))

# Evaluate the model
loss, accuracy = model.evaluate(X_val_reshaped, y_val)
print(f"Validation Loss: {loss:.2f}, Validation Accuracy: {accuracy:.2f}")

# Function to predict sentiment for new input text
def predict_sentiment(text):
    cleaned_text = clean_text(text)  # Clean the input text
    embedding = ft_model.get_sentence_vector(cleaned_text)
    reshaped_embedding = embedding.reshape((1, 1, -1))  # Reshape for LSTM input
    prediction = model.predict(reshaped_embedding)
    return (prediction[0][0] > 0.5)  # Returns True if positive, False if negative

# Example input
input_text = "Look ye politicians suvar jaise baithe rahteha in sirf money ke liye kaam karte hain. They don’t care about public."
predicted_label = predict_sentiment(input_text)

# Output the prediction
print(f"Predicted Label for the input text: {predicted_label}")


C:\Users\Acer\AppData\Local\Temp\ipykernel_10808\965962665.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Labels Set'] = df['Labels Set'].replace({'hate': 1, 'normal': 0})
C:\Users\Acer\Python37\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8769 - loss: 0.5694 - val_accuracy: 0.8696 - val_loss: 0.3227
Epoch 2/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9611 - loss: 0.1931 - val_accuracy: 0.8871 - val_loss: 0.3182
Epoch 3/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9662 - loss: 0.1211 - val_accuracy: 0.8865 - val_loss: 0.3619
Epoch 4/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9664 - loss: 0.1034 - val_accuracy: 0.8859 - val_loss: 0.3878
Epoch 5/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9651 - loss: 0.1078 - val_accuracy: 0.8859 - val_loss: 0.4041
Epoch 6/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9621 - loss: 0.1131 - val_accuracy: 0.8853 - val_loss: 0.4117
Epoch 7/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9672 - loss: 0.1035 - val_accuracy: 0.8859 - val_loss: 0.4147
Epoch 8/10
207/207 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9660 - loss: 0.1056 - val_accuracy: 0.

In [14]:
from sklearn.metrics import classification_report
import numpy as np

# Make predictions on the validation set
y_pred_probs = model.predict(X_val_reshaped)
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary labels (0 or 1)

# Generate the classification report
print(classification_report(y_val, y_pred, target_names=['Class 0', 'Class 1']))


52/52 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
              precision    recall  f1-score   support

     Class 0       0.90      0.91      0.91      1039
     Class 1       0.85      0.84      0.84       617

    accuracy                           0.89      1656
   macro avg       0.88      0.88      0.88      1656
weighted avg       0.88      0.89      0.89      1656



In [22]:
import re
import numpy as np
from sklearn.model_selection import train_test_split
import fasttext
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense, Dropout, BatchNormalization 
from tensorflow.keras.callbacks import EarlyStopping

# Clean the text function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Clean the text in the dataset
df['Post'] = df['Post'].apply(clean_text)

# Convert labels to 1 for 'hate' and 0 for 'normal'
df['Labels Set'] = df['Labels Set'].replace({'hate': 1, 'normal': 0})

# Split data into features and labels
X = df['Post']
y = np.array(df['Labels Set'], dtype=np.float32)

# Split dataset into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Save the training data to a temporary file for FastText
train_file = 'train.txt'
with open(train_file, 'w', encoding="utf-8") as f:
    for text, label in zip(X_train, y_train):
        f.write(f'__label__{int(label)} {text}\n')

# Train FastText model
ft_model = fasttext.train_supervised(train_file)

# Get embeddings for the training and validation sets
def get_fasttext_embeddings(texts, model):
    embeddings = []
    for text in texts:
        vec = model.get_sentence_vector(text)
        embeddings.append(vec)
    return np.array(embeddings)

# Generate embeddings
X_train_embeds = get_fasttext_embeddings(X_train, ft_model)
X_val_embeds = get_fasttext_embeddings(X_val, ft_model)

# Reshape embeddings for 1D CNN input
X_train_reshaped = X_train_embeds.reshape((X_train_embeds.shape[0], 1, X_train_embeds.shape[1]))
X_val_reshaped = X_val_embeds.reshape((X_val_embeds.shape[0], 1, X_val_embeds.shape[1]))

# Build 1D CNN model
# Build 1D CNN model with adjusted kernel size
from tensorflow.keras.callbacks import EarlyStopping

# Build a more robust 1D CNN model with additional layers and regularization
model = Sequential()
model.add(Conv1D(filters=128, kernel_size=1, activation='relu', input_shape=(X_train_reshaped.shape[1], X_train_reshaped.shape[2])))
model.add(BatchNormalization())  # Batch normalization for regularization
model.add(Dropout(0.5))  # Dropout for preventing overfitting

# Add an additional Conv1D layer
model.add(Conv1D(filters=64, kernel_size=1, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

# Add GlobalMaxPooling1D to reduce the dimensions
model.add(GlobalMaxPooling1D())

# Add a Dense layer
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))  # Dropout layer

# Output layer
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Set up EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model with early stopping
history = model.fit(
    X_train_reshaped, y_train,
    epochs=20,  # Increase epochs if needed
    batch_size=32,  # Adjust batch size as needed
    validation_data=(X_val_reshaped, y_val),
    callbacks=[early_stopping]
)

# Evaluate the model
loss, accuracy = model.evaluate(X_val_reshaped, y_val)
print(f"Validation Loss: {loss:.2f}, Validation Accuracy: {accuracy:.2f}")


# Function to predict sentiment for new input text
def predict_sentiment(text):
    cleaned_text = clean_text(text)
    embedding = ft_model.get_sentence_vector(cleaned_text)
    reshaped_embedding = embedding.reshape((1, 1, -1))  # Reshape for CNN input
    prediction = model.predict(reshaped_embedding)
    return (prediction[0][0] > 0.5)  # Returns True if positive, False if negative

# Example input
input_text = "Look ye politicians suvar jaise baithe rahteha in sirf money ke liye kaam karte hain. They don’t care about public."
predicted_label = predict_sentiment(input_text)

# Output the prediction
print(f"Predicted Label for the input text: {predicted_label}")


Epoch 1/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8619 - loss: 0.3260 - val_accuracy: 0.7301 - val_loss: 0.4945
Epoch 2/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9440 - loss: 0.1651 - val_accuracy: 0.8527 - val_loss: 0.3302
Epoch 3/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9490 - loss: 0.1538 - val_accuracy: 0.8798 - val_loss: 0.3451
Epoch 4/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9517 - loss: 0.1483 - val_accuracy: 0.8780 - val_loss: 0.4467
Epoch 5/20
207/207 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9573 - loss: 0.1369 - val_accuracy: 0.8835 - val_loss: 0.4300
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8613 - loss: 0.3175
Validation Loss: 0.33, Validation Accuracy: 0.85
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step
Predicted Label for the input text: False


In [17]:
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D


In [15]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size,
                    output_dim=embedding_dim,
                    weights=[embedding_matrix],
                    input_length=max_length,
                    trainable=False))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

# Evaluate the model
loss, accuracy = model.evaluate(X_val, y_val)
print(f'Validation Accuracy: {accuracy:.2f}')
        


NameError: name 'Embedding' is not defined

In [ ]:
import numpy as np
import pandas as pd
import gensim
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

from sklearn.model_selection import train_test_split

In [ ]:
import numpy as np
import pandas as pd
import gensim
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, Dense, Embedding, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences
from sklearn.model_selection import train_test_split

# Load FastText model (ensure the FastText embeddings are already trained and available)
fasttext_model_path = 'path/to/your/fasttext_model.bin'
fasttext_model = gensim.models.KeyedVectors.load_word2vec_format(fasttext_model_path, binary=True)

# Load your dataset (assuming a CSV with columns 'text' and 'label')
data_path = 'path/to/your/dataset.csv'
df = pd.read_csv(data_path)

# Prepare the data
texts = df['text'].astype(str).values
labels = df['label'].values

# Tokenize and pad the sequences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)
vocab_size = len(tokenizer.word_index) + 1
max_length = 100  # Adjust as needed

X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X, maxlen=max_length)
y = np.array(labels)

# Create FastText embedding matrix
embedding_dim = fasttext_model.vector_size
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in fasttext_model:
        embedding_matrix[i] = fasttext_model[word]

# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Build the CNN model
model = Sequential()
model.add(Embedding(input_dim=vocab_size,
                    output_dim=embedding_dim,
                    weights=[embedding_matrix],
                    input_length=max_length,
                    trainable=False))
model.add(Conv1D(filters=128, kernel_size=5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))

# Evaluate the model
loss, accuracy = model.evaluate(X_val, y_val)
print(f'Validation Accuracy: {accuracy:.2f}')
